In [34]:
from tqdm import tqdm

In [35]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())

CUDA available: True
Device count: 1


In [36]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "0"

In [37]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [38]:
from unsloth.chat_templates import get_chat_template

In [39]:
HF_TOKEN = os.getenv("HF_TOKEN")
print(len(HF_TOKEN))

37


In [40]:
from huggingface_hub import login
login(HF_TOKEN)

In [41]:
MODEL_ID = "unsloth/gemma-3-270m-bnb-4bit"

In [42]:
from unsloth import FastLanguageModel

In [43]:
model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_ID,
    dtype=None, # Tự động chọn float16 hoặc bfloat16 tùy theo phần cứng GPU
    max_seq_length=512
)

==((====))==  Unsloth 2026.8.15: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3050 Laptop GPU. Num GPUs = 1. Max memory: 4.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.13.0+cu132. CUDA: 8.6. CUDA Toolkit: 13.2. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

In [44]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-3",
)

In [45]:
def format_prompt(examples, tokenizer):
    texts = [tokenizer.apply_chat_template(x, add_generation_prompt=False, tokenize=False) for x in examples['messages']]
    return {"text": texts}

In [46]:
from datasets import load_dataset

DATASET_ID = "HuggingFaceH4/ultrachat_200k"
dataset = load_dataset(DATASET_ID)
print(f"Colums: {dataset.column_names}")

Colums: {'train_sft': ['prompt', 'prompt_id', 'messages'], 'test_sft': ['prompt', 'prompt_id', 'messages'], 'train_gen': ['prompt', 'prompt_id', 'messages'], 'test_gen': ['prompt', 'prompt_id', 'messages']}


In [47]:
dataset = dataset["train_sft"].remove_columns(["prompt", "prompt_id"])

In [48]:
dataset = dataset.select(range(2000)).map(format_prompt, batched=True, fn_kwargs={"tokenizer": tokenizer})

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

# Gắn LoRA

In [49]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules=['q_proj', 'v_proj'],
    lora_alpha=16,
    lora_dropout=0.05, # để 0.05 sẽ train chậm hơn chút so với để 0 do cấu hình của unsloth
)

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


In [50]:
from trl import SFTTrainer, SFTConfig

In [51]:
sft_conf = SFTConfig(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    logging_steps=1,
    learning_rate=2e-4,
    num_train_epochs=1,
    packing = True,
    max_length=128
)

trainer = SFTTrainer(
    model=model,
    tokenizer = tokenizer,
    train_dataset=dataset,
    args=sft_conf
)

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"]:   0%|          | 0/2000 [00:00<?, ? examples/s]

Unsloth: Packing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


In [52]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,992 | Num Epochs = 1 | Total steps = 249
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 368,640 of 268,466,816 (0.14% trained)
Autotune Choices Stats:
{"num_choices": 10, "num_triton_choices": 10, "best_kernel": "triton_flex_attention_backward_96", "best_kernel_desc": "BLOCKS_ARE_CONTIGUOUS=False, BLOCK_M1=32, BLOCK_M2=32, BLOCK_N1=32, BLOCK_N2=32, FLOAT32_PRECISION=\"'ieee'\", GQA_SHARED_HEADS=4, HAS_FULL_BLOCKS=True, IS_DIVISIBLE=False, OUTPUT_LOGSUMEXP=True, OUTPUT_MAX=False, PRESCALE_QK=False, QK_HEAD_DIM=256, QK_HEAD_DIM_ROUNDED=256, ROWS_GUARANTEED_SAFE=False, SAFE_HEAD_DIM=True, SM_SCALE=0.0625, SPARSE_KV_BLOCK_SIZE=128, SPARSE_Q_BLOCK_SIZE=128, USE_TMA=False, V_HEAD_DIM=256, V_HEAD_DIM_ROUNDED=256, WRITE_DQ=True, num_stages=2, num_warps=4", "bes

Step,Training Loss
1,2.345322
2,2.748871
3,2.356826
4,2.476571
5,2.820608
6,2.698880
7,2.712870
8,2.514007
9,2.713332
10,2.551584


TrainOutput(global_step=249, training_loss=10.317652600836084, metrics={'train_runtime': 284.0929, 'train_samples_per_second': 7.012, 'train_steps_per_second': 0.876, 'total_flos': 607795360287744.0, 'train_loss': 10.317652600836084, 'epoch': 1.0})

In [53]:
model.save_pretrained("gemma-3-270m-bnb-4bit-finetuned-ultrachat-2k")
tokenizer.save_pretrained("gemma-3-270m-bnb-4bit-finetuned-ultrachat-2k")

Unsloth: Restored added_tokens_decoder metadata in gemma-3-270m-bnb-4bit-finetuned-ultrachat-2k\tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in gemma-3-270m-bnb-4bit-finetuned-ultrachat-2k.


('gemma-3-270m-bnb-4bit-finetuned-ultrachat-2k\\tokenizer_config.json',
 'gemma-3-270m-bnb-4bit-finetuned-ultrachat-2k\\chat_template.jinja',
 'gemma-3-270m-bnb-4bit-finetuned-ultrachat-2k\\tokenizer.json')